# GraviLens v2 driver

Thin Colab driver only. All simulation, training, evaluation, and audit logic lives in the `gravilens` package.

Set `REPO` and `TAG` below. The tag pins the exact audited code: push it with
`git push origin main --tags`. Bump `TAG` (and the tag itself) if the package changes.
A GPU runtime is strongly recommended for the ablation; data generation is CPU-bound.

In [ ]:
import os
REPO = "https://github.com/<handle>/gravilens"
TAG = "v0.1.0"
if not os.path.isdir('/content/gravilens'):
    !git clone {REPO} /content/gravilens
%cd /content/gravilens
!git fetch --tags && git checkout {TAG}
!pip -q install -e ".[sim,dev]"

In [ ]:
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## Generate the matched dataset

The Galaxy10 DECals file is downloaded once (~2.5 GB). A triplet costs ~8 s on CPU, so the
2,333-triplet train split is ~5 CPU-hours; `val` and `test` run concurrently with it.
Shards are resumable, so rerunning the cell continues where it stopped. Skip this cell if
`data/gravilens-v2` already holds all three splits.

In [ ]:
%%bash
python -m gravilens generate --config configs/data.yaml --splits train > train_gen.log 2>&1 &
python -m gravilens generate --config configs/data.yaml --splits val   > val_gen.log   2>&1 &
python -m gravilens generate --config configs/data.yaml --splits test  > test_gen.log  2>&1 &
wait
tail -n 2 train_gen.log val_gen.log test_gen.log

## Smoke test (2 epochs)

This checks the full train/evaluate path before spending GPU time on the 21-run grid.
The smoke artifacts go to `runs-smoke/`, not `runs/`, so they can never be mistaken for
grid results.

In [ ]:
!python -m gravilens train --config configs/ablation.yaml --model gravilens --variant full --seed 42 --epochs 2 --out runs-smoke/gravilens_full_seed42
!python -m gravilens evaluate --run runs-smoke/gravilens_full_seed42

## Full ablation (21 runs)

4 variants × 3 seeds + 3 parameter-matched baselines × 3 seeds. `ablate` skips a run only
when it used the same recipe and the same dataset, and redoes it otherwise. `audit` exits
non-zero until all 21 runs exist and every run used the pre-registered recipe;
`figures` writes the main table and figures from the artifacts only.

Runtimes are ephemeral: rerun the cell to continue (finished runs are skipped), or split
the grid, e.g. `ablate --variants full no_sup no_sis` and then
`ablate --baselines vit cait vitsd`. Copy `runs/` to Drive before disconnecting.

In [ ]:
!python -m gravilens ablate --config configs/ablation.yaml
!python -m gravilens audit
!python -m gravilens figures

## Save artifacts

Colab runtimes are ephemeral. Copy `runs/` to Drive or download the run directories before disconnecting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r runs /content/drive/MyDrive/gravilens-runs